Extract Charlson Comorbidity Index (time-varying) for all patients.

# Setup

In [ ]:
library(tidyverse)
library(bigrquery)
library(survival)
library(lubridate)
library(tictoc)

In [ ]:
source("functions.R")

In [ ]:
tic("Time to run script")

# Pull raw conditions data

In [ ]:
library(tidyverse)
library(bigrquery)

CCI_sql <- paste("
    SELECT
        c_occurrence.condition_concept_id,
        c_occurrence.condition_start_datetime condition_start_date,
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name 
    FROM
        ( SELECT
            * 
        from
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN  (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            select
                                cast(cr.id as string) as id 
                            FROM
                                `cb_criteria` cr 
                            WHERE
                                concept_id IN (
                                    134442, 192606, 192680, 201820, 24966, 255348, 256197, 257628, 316139, 321052,
                                    374022, 381591, 4029488, 4030518, 4063381, 4064161, 4182210, 4212540, 4245975, 4247120, 
                                    432851, 4329847, 434056, 43531586, 442793, 443392, 443767, 80800, 80809
                                ) 
                                AND full_text LIKE '%_rank1]%'
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )
            ) c_occurrence 
        left join
            `concept` c_standard_concept 
                on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
CCI_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  ##strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_78756530",
  "CCI_*.csv")

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), CCI_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  CCI_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_78756530_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

CCI_df <- read_bq_export_from_workspace_bucket(CCI_path)
dim(CCI_df)

# Get the ancestor concept id assignments

In [ ]:
subquery_sql <- paste("
                    SELECT
                        * 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            select
                                cast(cr.id as string) as id, concept_id as INPUT_CONCEPT_ID
                            FROM
                                `cb_criteria` cr 
                            WHERE
                                concept_id IN (
                                    134442, 192606, 192680, 201820, 24966, 255348, 256197, 257628, 316139, 321052,
                                    374022, 381591, 4029488, 4030518, 4063381, 4064161, 4182210, 4212540, 4245975, 4247120, 
                                    432851, 4329847, 434056, 43531586, 442793, 443392, 443767, 80800, 80809
                                ) 
                                AND full_text LIKE '%_rank1]%'
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        
              ", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
subquery_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  ##strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_subquery",
  "condition_subquery_*.csv")

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), subquery_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  subquery_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_78756530_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

subquery_df <- read_bq_export_from_workspace_bucket(subquery_path)
dim(subquery_df)

In [ ]:
head(subquery_df, 3) %>% 
    select(-c(domain_id, is_standard, type, subtype, value, is_group, is_selectable, has_attribute,
              has_hierarchy, synonyms, full_text, display_synonyms, rollup_count, item_count, 
              est_count, path, id_1))

In [ ]:
cond_map <- data.frame(
    INPUT_CONCEPT_ID = c(
            4329847,
            316139,
            321052,
            381591,
            434056,
            4182210,
            4063381,
            257628,
            134442,
            80800,
            80809,
            256197,
            255348,
            4247120,
            4064161,
            4212540,
            201820,
            443767,
            442793,
            192606,
            374022,
            4030518,
            443392,
            4245975,
            4029488,
            192680,
            24966,
            432851,
            43531586), 
    condition = c(
            "Myocardial Infarction",
            "Congestive Heart Failure",
            "Peripheral Vascular Disease",
            "Cerebrovascular Disease",
            "Cerebrovascular Disease",
            "Dementia",
            "Chronic Pulmonary Disease",
            "Rheumatologic Disease",
            "Rheumatologic Disease",
            "Rheumatologic Disease",
            "Rheumatologic Disease",
            "Rheumatologic Disease",
            "Rheumatologic Disease",
            "Peptic ulcer disease",
            "Mild Liver Disease",
            "Mild Liver Disease",
            "Diabetes Mild to Moderate",
            "Diabetes with chronic complications",
            "Diabetes with chronic complications",
            "Hemoplegia or Paraplegia",
            "Hemoplegia or Paraplegia",
            "Renal Disease",
            "Any Malignancy",
            "Moderate to severe liver disease",
            "Moderate to severe liver disease",
            "Moderate to severe liver disease",
            "Moderate to severe liver disease",
            "Metastatic solid tumour",
            "AIDS"),
    weight = c(
            1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 
            2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 6, 6
      )
    ) %>%
    full_join(subquery_df %>% select(INPUT_CONCEPT_ID, condition_concept_id = concept_id))

cond_map

In [ ]:
CCI_df <- CCI_df %>%
    mutate(condition_start_date = as.Date(condition_start_date)) %>%
    select(-standard_concept_name) %>%
    distinct() 

In [ ]:
gc()

# Categorize the raw conditions data

In [ ]:
CCI_dat <- cond_map %>% distinct(condition_concept_id, condition, weight) %>%
    inner_join(CCI_df) %>%
    arrange(person_id, condition, condition_start_date) %>%
    distinct()
dim(CCI_dat)

In [ ]:
CCI_first <- CCI_dat %>%
    group_by(person_id, condition, weight) %>%
    summarize(first_occurrence = min(condition_start_date))

dim(CCI_first)

In [ ]:
length(unique(CCI_first$person_id))

In [ ]:
rm(CCI_df, subquery_df, CCI_dat)
gc()

# Create the tmerge object

In [ ]:
#Use tmerge to get the time-dependent dataset.  Use arbitrary start and endpoints, then convert back to date. 

#arbitrary origin
ogn <- min(as.Date(CCI_first$first_occurrence))
ogn

In [ ]:
#Initialize the tmerge object 

tmp <-  CCI_first %>% ungroup() %>% distinct(person_id) %>% mutate(time = 1000, status = 0) %>% arrange(person_id)

tindpt <- tmerge(tmp, tmp, id = person_id, endpt = event(time, status))

head(tindpt %>% select(-person_id))

In [ ]:
tdept <- slip_condition_first("Myocardial Infarction")

In [ ]:
tic()
for (condition in unique(cond_map$condition)[-1]) {
    print(condition)
    tdept <- slip_condition(condition)
}
toc()

In [ ]:
# Use the first visit to help fill in tstart
firstvisit_sql <- "SELECT 
        c_occurrence.person_id,
        MIN(CAST(c_occurrence.CONDITION_START_DATETIME AS DATE)) AS first_condition_date
    FROM `condition_occurrence` c_occurrence
    GROUP BY PERSON_ID"

firstvisit_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "handwritten",
  "first_condition_*.csv")

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), firstvisit_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  firstvisit_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

firstvisit_df <- read_bq_export_from_workspace_bucket(firstvisit_path)

dim(firstvisit_df) #

In [ ]:
write_to_bucket(firstvisit_df, "first_condition_any.csv")

In [ ]:
tdept_cleaned <- tdept %>% select(-c(time, status, tstop, endpt)) %>%
    full_join(firstvisit_df)
tdept_cleaned[is.na(tdept_cleaned)] <- 0

nrow(tdept_cleaned)
length(unique(tdept_cleaned$person_id))

# Convert time back to date format

In [ ]:
tdept_cleaned <- tdept_cleaned %>%
    mutate(tstart = date_decimal(decimal_date(ogn) + tstart),
           #below line only replaces initial T0 with true first observation time. Later rows not affected
          tstart = pmax(tstart, first_condition_date)) %>% 
    mutate(tstart = as.Date(tstart))

In [ ]:
tdept_cleaned_summary_CCI <- tdept_cleaned %>%
    select(person_id, tstart, ends_with("weight")) 

tdept_cleaned_summary_CCI$CCI <- rowSums(tdept_cleaned_summary_CCI %>% 
                                         select(-c(person_id, tstart)), na.rm = T)

In [ ]:
tdept_CCI_excl_renal <- tdept_cleaned %>%
    select(person_id, tstart, ends_with("weight")) %>%
    select(-Renal_Disease_weight)

tdept_cleaned_summary_CCI$CCI_excl_renal <- rowSums(tdept_CCI_excl_renal %>% 
                                                    select(-c(person_id, tstart)), na.rm = T)

In [ ]:
# this includes all the factors, not just CCI
write_to_bucket(tdept_cleaned_summary_CCI, "CCI_time_dependent_all.csv")

# Finalize the output

In [ ]:
CCI_out <- tdept_cleaned_summary_CCI %>%
    select(person_id, tstart, CCI, CCI_excl_renal)

In [ ]:
write_to_bucket(CCI_out, "CCI_time_dependent_score.csv")

In [ ]:
toc()

In [ ]:
rm(list = ls())
gc()